# Social Media Market Sentiment NLP Classification

**Author:** Ahmed Noureldin  
**Affiliation:** Accounting Scholar (Year 3), New Cairo Higher Institute | Ex-CIB Data Analytics Intern  
**Email:** ahmedn4474@gmail.com  
**Domain:** Natural Language Processing (NLP), Sentiment Analysis & Text Mining  
**Dataset:** Sentiment140 Balanced Corpus (50,000 Sampled & Preprocessed Social Media Posts)

---

## 1. Executive Summary & NLP Formulation

Sentiment classification from unstructured social media streams enables tracking consumer sentiment, brand perception, and real-time market mood. This project builds a TF-IDF and regularized Logistic Regression classifier with text cleaning, tokenization, and evaluation.


## 2. Library Imports & Configuration


In [ ]:
import os, glob, re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
print('NLP Libraries loaded successfully.')


## 3. Data Ingestion & Preprocessing


In [ ]:
candidate_paths = [
    os.path.join("data", "sentiment140_sample.csv"),
    os.path.join("..", "data", "sentiment140_sample.csv"),
    os.path.join(".", "sentiment140_sample.csv"),
] + glob.glob("/kaggle/input/**/training*1600000*.csv", recursive=True) + glob.glob("/kaggle/input/**/*sentiment*.csv", recursive=True)

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if not data_path:
    k_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if k_files: data_path = k_files[0]

print(f"Loading Sentiment Data from: {data_path}")
df_raw = pd.read_csv(data_path)
if "target" in df_raw.columns:
    df_raw["sentiment"] = (df_raw["target"].astype(str).str.strip() == "4").astype(int)
elif "sentiment" not in df_raw.columns:
    df_raw["sentiment"] = (df_raw.iloc[:, 0].astype(str).str.strip() == "4").astype(int)

text_col = "text" if "text" in df_raw.columns else df_raw.columns[-1]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

sample_df = df_raw.dropna(subset=[text_col]).copy()
sample_df["clean_text"] = sample_df[text_col].apply(clean_text)
sample_df["char_length"] = sample_df["clean_text"].apply(len)
sample_df["word_count"] = sample_df["clean_text"].apply(lambda x: len(x.split()))
sample_df = sample_df[sample_df["clean_text"].str.len() > 2].reset_index(drop=True)
print(f"Sample Corpus Shape: {sample_df.shape[0]:,d} cleaned tweets")
print(f"Class Balance: Negative (0) = {(sample_df['sentiment']==0).sum():,d} | Positive (1) = {(sample_df['sentiment']==1).sum():,d}")
print(sample_df[["clean_text", "sentiment"]].head(3))


## 4. Exploratory Text & Length Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=sample_df, x='sentiment', palette=['#d62728', '#2ca02c'], ax=axes[0])
axes[0].set_title('Sentiment Class Distribution (0=Neg, 1=Pos)', fontweight='bold')

sns.kdeplot(data=sample_df, x='char_length', hue='sentiment', palette=['#d62728', '#2ca02c'], ax=axes[1])
axes[1].set_title('Tweet Character Length Distribution', fontweight='bold')
plt.tight_layout()
plt.show()


## 5. TF-IDF Vectorization & Classifier Training


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    sample_df['clean_text'], sample_df['sentiment'],
    test_size=0.20, stratify=sample_df['sentiment'], random_state=42
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

clf = LogisticRegression(C=1.5, max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)
y_probs = clf.predict_proba(X_test_vec)[:, 1]
y_preds = clf.predict(X_test_vec)

print(f'Test ROC-AUC: {roc_auc_score(y_test, y_probs):.4f} | PR-AUC: {average_precision_score(y_test, y_probs):.4f}')
print(classification_report(y_test, y_preds))


## 6. Evaluation & Top Predictive N-Grams


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix Heatmap', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

feat_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]
top_pos_idx = np.argsort(coefs)[-10:]
axes[1].barh(feat_names[top_pos_idx], coefs[top_pos_idx], color='#2ca02c')
axes[1].set_title('Top 10 Positive Sentiment Predictors', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Real-Time Inference Function & Takeaways

1. **Sub-word and N-gram Context:** Bi-grams ('not good', 'really happy') provide critical contextual grounding compared to unigram bag-of-words.
2. **Linear Efficiency:** Regularized L2 Logistic Regression provides sub-millisecond inference per tweet suitable for live stream processing.

---

*Authored by Ahmed Noureldin Mohamed — Applied Quantitative & NLP Analytics.*